# 01 · Calidad y modelo financiero

Comprueba integridad, granularidad y reconciliación del P&L antes de analizar resultados.

> **Fuente:** datos sintéticos de Levante Ferries. Proyecto demostrativo; no contiene información real de ninguna naviera.

## Contexto y método

El notebook forma parte de una cadena reproducible. Las fórmulas y supuestos se muestran junto a los resultados para que cada conclusión pueda revisarse.

In [1]:
from pathlib import Path
import json, math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Markdown

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
RAW = ROOT / 'data' / 'raw'
PROCESSED = ROOT / 'data' / 'processed'
TABLES = ROOT / 'outputs' / 'tables'
FIGURES = ROOT / 'outputs' / 'figures'
for folder in [PROCESSED, TABLES, FIGURES]: folder.mkdir(parents=True, exist_ok=True)
sns.set_theme(style='whitegrid', palette=['#1f6feb','#2dd4bf','#f59e0b','#ef4444','#94a3b8'])
plt.rcParams.update({'figure.figsize': (11, 5.5), 'axes.titlesize': 14, 'axes.labelsize': 10})
pd.options.display.float_format = '{:,.2f}'.format


## Carga y controles

In [2]:
actual=pd.read_csv(RAW/'fact_finance_actual.csv',parse_dates=['month'])
checks=[]
def add(name,passed,detail): checks.append({'check':name,'passed':bool(passed),'detail':detail})
add('Clave mes-ruta única',not actual.duplicated(['month','route_id']).any(),'252 combinaciones esperadas')
add('Sin nulos críticos',not actual[['revenue','variable_cost','fixed_allocated','ebitda']].isna().any().any(),'Campos financieros completos')
recalc=actual.revenue-actual.variable_cost-actual.fixed_allocated
add('EBITDA reconcilia',np.allclose(recalc,actual.ebitda,rtol=0,atol=.02),'Ingresos - variable - fijo')
add('Ocupación válida',actual.occupancy.between(0,1).all(),'Rango 0–100%')
quality=pd.DataFrame(checks)
quality.to_csv(TABLES/'01_quality_checks.csv',index=False)
display(quality)
assert quality.passed.all()

                  check  passed                        detail
0  Clave mes-ruta única    True   252 combinaciones esperadas
1    Sin nulos críticos    True  Campos financieros completos
2     EBITDA reconcilia    True    Ingresos - variable - fijo
3      Ocupación válida    True                  Rango 0–100%


## Modelo analítico

In [3]:
model=pd.DataFrame({'table':['fact_finance_actual','fact_finance_budget','fact_cashflow_monthly','dim_route','dim_vessel'],'grain':['mes × ruta','mes × ruta','mes','ruta','buque'],'role':['real P&L y volumen','plan y desviaciones','caja y circulante','geografía/distancia','capacidad/eficiencia']})
model.to_csv(TABLES/'01_model_catalog.csv',index=False)
display(model)

                   table       grain                  role
0    fact_finance_actual  mes × ruta    real P&L y volumen
1    fact_finance_budget  mes × ruta   plan y desviaciones
2  fact_cashflow_monthly         mes     caja y circulante
3              dim_route        ruta   geografía/distancia
4             dim_vessel       buque  capacidad/eficiencia


## Conclusiones

Las conclusiones concretas se generan a partir de las salidas ejecutadas. Deben interpretarse como evidencia de una simulación y como demostración del método analítico.